In [1]:
import pandas as pd 
df = pd.read_csv('../data/processed/modelB_dataset.csv')

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   dyad                      1440 non-null   str    
 1   year                      1440 non-null   int64  
 2   regime_diff               1440 non-null   float64
 3   MonthYear                 1440 non-null   int64  
 4   event_count               1440 non-null   float64
 5   goldstein_std             1440 non-null   float64
 6   goldstein_min             1440 non-null   float64
 7   num_mentions_sum          1440 non-null   float64
 8   num_articles_sum          1440 non-null   float64
 9   num_sources_sum           1440 non-null   float64
 10  high_conflict_count       1440 non-null   float64
 11  low_conflict_count        1440 non-null   float64
 12  quad4_count               1440 non-null   float64
 13  high_conflict_pct         1440 non-null   float64
 14  low_conflict_pct   

In [2]:
# define data
X = df.loc[:, ['regime_diff','event_count_lag1', 'goldstein_std_lag1', 'goldstein_min_lag1', 'num_mentions_sum_lag1',
               'num_articles_sum_lag1', 'num_sources_sum_lag1', 'high_conflict_count_lag1', 'low_conflict_count_lag1',
               'quad4_count_lag1', 'high_conflict_pct_lag1', 'low_conflict_pct_lag1', 'quad4_pct_lag1']]
y = df['monthly_label']

# 先按時間一刀切
train_mask = df['MonthYear'] < 202301
test_mask =  df['MonthYear'] >= 202301
X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

In [6]:
# lgb model
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn import set_config
set_config(display='text')
lgb_params = {
    'objective': 'multiclass',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,       
    'num_leaves': 7,             
    'max_depth': 3,               
    'min_child_samples': 20,      
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,             
    'reg_lambda': 0.1,            
    'random_state': 42,
}
mod1 = LGBMClassifier(**lgb_params)
mod1.fit(X_train, y_train,
         sample_weight=compute_sample_weight('balanced',  y_train))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002354 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2886
[LightGBM] [Info] Number of data points in the train set: 1044, number of used features: 13
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, max_depth=3,
               num_leaves=7, objective='multiclass', random_state=42,
               reg_alpha=0.1, reg_lambda=0.1, subsample=0.8)

In [9]:
from sklearn.metrics import classification_report, roc_auc_score

predict_train = mod1.predict(X_train)
predict_test = mod1.predict(X_test)
report_train = classification_report(y_train, predict_train)
report_test = classification_report(y_test, predict_test)
print(f'train: {report_train}')
print(f'test: {report_test}')
# auc
y_pred_proba = mod1.predict_proba(X_test)  
auc_macro = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
print(f'Macro AUC: {auc_macro:.3f}')

#　個別
from sklearn.preprocessing import label_binarize
classes = mod1.classes_
y_test_binarized = label_binarize(y_test, classes=classes)
for i, cls in enumerate(classes):
    auc = roc_auc_score(y_test_binarized[:, i], y_pred_proba[:, i])
    print(f'{cls} AUC: {auc:.3f}')


train:                precision    recall  f1-score   support

  Cooperation       0.96      0.87      0.91       777
High_Conflict       0.71      0.95      0.81        95
 Low_Conflict       0.64      0.81      0.71       172

     accuracy                           0.86      1044
    macro avg       0.77      0.87      0.81      1044
 weighted avg       0.89      0.86      0.87      1044

test:                precision    recall  f1-score   support

  Cooperation       0.93      0.92      0.93       244
High_Conflict       0.56      0.61      0.59        44
 Low_Conflict       0.72      0.72      0.72       108

     accuracy                           0.83       396
    macro avg       0.74      0.75      0.74       396
 weighted avg       0.83      0.83      0.83       396

Macro AUC: 0.894
Cooperation AUC: 0.944
High_Conflict AUC: 0.888
Low_Conflict AUC: 0.850


In [11]:
import pandas as pd

importance_df = pd.DataFrame({
    'feature': mod1.feature_name_,
    'gain': mod1.booster_.feature_importance(importance_type='gain')
}).sort_values('gain', ascending=False)

print(importance_df)

                     feature         gain
0                regime_diff  5848.255845
11     low_conflict_pct_lag1  4217.782484
10    high_conflict_pct_lag1  2458.188159
2         goldstein_std_lag1  1110.509124
12            quad4_pct_lag1   815.677218
8    low_conflict_count_lag1   571.416836
7   high_conflict_count_lag1   297.733862
4      num_mentions_sum_lag1   266.366418
9           quad4_count_lag1   237.564881
6       num_sources_sum_lag1   215.309913
1           event_count_lag1   198.854423
5      num_articles_sum_lag1   138.198260
3         goldstein_min_lag1     2.819430


In [20]:
from sklearn.model_selection import LeaveOneGroupOut
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import recall_score
import numpy as np

logo = LeaveOneGroupOut()
groups = df['dyad']

lgb_params = {
    'objective': 'multiclass',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 7,
    'max_depth': 3,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

results = []

for train_idx, test_idx in logo.split(X, y, groups):
    left_out_dyad = groups.iloc[test_idx].unique()[0]
    
    X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
    y_train_fold, y_test_fold = y.iloc[train_idx], y.iloc[test_idx]
    
    model = LGBMClassifier(**lgb_params)
    model.fit(X_train_fold, y_train_fold, sample_weight=compute_sample_weight('balanced', y_train_fold))
    
    y_pred_fold = model.predict(X_test_fold)
    
    recall_macro = recall_score(y_test_fold, y_pred_fold, average='macro', zero_division=0)
    
    results.append({'left_out_dyad': left_out_dyad, 'recall_macro': recall_macro})
    print(f'{left_out_dyad}: recall_macro = {recall_macro:.3f}')

results_df = pd.DataFrame(results)
print(results_df)
print(f'平均 recall_macro: {results_df["recall_macro"].mean():.3f}')

CHN-JPN: recall_macro = 0.461
CHN-KOR: recall_macro = 0.310
CHN-PHL: recall_macro = 0.498
CHN-PRK: recall_macro = 0.333
CHN-TWN: recall_macro = 0.695
CHN-VNM: recall_macro = 0.642
JPN-KOR: recall_macro = 0.318
JPN-PRK: recall_macro = 0.459
JPN-TWN: recall_macro = 0.496
KOR-PRK: recall_macro = 0.541
KOR-TWN: recall_macro = 0.311
   left_out_dyad  recall_macro
0        CHN-JPN      0.460979
1        CHN-KOR      0.310256
2        CHN-PHL      0.498176
3        CHN-PRK      0.333333
4        CHN-TWN      0.694677
5        CHN-VNM      0.642077
6        JPN-KOR      0.317829
7        JPN-PRK      0.459259
8        JPN-TWN      0.496124
9        KOR-PRK      0.541453
10       KOR-TWN      0.311475
平均 recall_macro: 0.461


In [22]:
lag_cols = ['event_count_lag1', 'goldstein_std_lag1', 'goldstein_min_lag1', 'num_mentions_sum_lag1',
           'num_articles_sum_lag1', 'num_sources_sum_lag1', 'high_conflict_count_lag1', 'low_conflict_count_lag1',
           'quad4_count_lag1', 'high_conflict_pct_lag1', 'low_conflict_pct_lag1', 'quad4_pct_lag1']
X_no_regime = df[lag_cols]  

results_no_regime = []

for train_idx, test_idx in logo.split(X_no_regime, y, groups):
    left_out_dyad = groups.iloc[test_idx].unique()[0]
    
    X_train_fold, X_test_fold = X_no_regime.iloc[train_idx], X_no_regime.iloc[test_idx]
    y_train_fold, y_test_fold = y.iloc[train_idx], y.iloc[test_idx]
    
    model = LGBMClassifier(**lgb_params)
    model.fit(X_train_fold, y_train_fold, sample_weight=compute_sample_weight('balanced', y_train_fold))
    
    y_pred_fold = model.predict(X_test_fold)
    recall_macro = recall_score(y_test_fold, y_pred_fold, average='macro', zero_division=0)
    
    results_no_regime.append({'left_out_dyad': left_out_dyad, 'recall_macro': recall_macro})
    print(f'{left_out_dyad}: recall_macro = {recall_macro:.3f}')

results_no_regime_df = pd.DataFrame(results_no_regime)
print(f'平均 recall_macro（無 regime_diff）: {results_no_regime_df["recall_macro"].mean():.3f}')

CHN-JPN: recall_macro = 0.472
CHN-KOR: recall_macro = 0.651
CHN-PHL: recall_macro = 0.471
CHN-PRK: recall_macro = 0.470
CHN-TWN: recall_macro = 0.467
CHN-VNM: recall_macro = 0.353
JPN-KOR: recall_macro = 0.310
JPN-PRK: recall_macro = 0.509
JPN-TWN: recall_macro = 0.326
KOR-PRK: recall_macro = 0.540
KOR-TWN: recall_macro = 0.298
平均 recall_macro（無 regime_diff）: 0.442


In [26]:
# 排除 goldstein_min_lag1
features = ['regime_diff', 'event_count_lag1', 'goldstein_std_lag1', 'goldstein_min_lag1', 'num_mentions_sum_lag1',
           'num_articles_sum_lag1', 'num_sources_sum_lag1', 'high_conflict_count_lag1', 'low_conflict_count_lag1',
           'quad4_count_lag1', 'high_conflict_pct_lag1', 'low_conflict_pct_lag1', 'quad4_pct_lag1']
lag_cols_trimmed = [col for col in features if col != 'goldstein_min_lag1']
print(lag_cols_trimmed)

X_trimmed = df[lag_cols_trimmed]
y = df['monthly_label']

X_train_trimmed = X_trimmed[train_mask]
X_test_trimmed = X_trimmed[test_mask]

mod_trimmed = LGBMClassifier(**lgb_params)
mod_trimmed.fit(X_train_trimmed, y_train, sample_weight=compute_sample_weight('balanced', y_train))

pred_train_trimmed = mod_trimmed.predict(X_train_trimmed)
pred_test_trimmed = mod_trimmed.predict(X_test_trimmed)

print('train:', classification_report(y_train, pred_train_trimmed))
print('test:', classification_report(y_test, pred_test_trimmed))


['regime_diff', 'event_count_lag1', 'goldstein_std_lag1', 'num_mentions_sum_lag1', 'num_articles_sum_lag1', 'num_sources_sum_lag1', 'high_conflict_count_lag1', 'low_conflict_count_lag1', 'quad4_count_lag1', 'high_conflict_pct_lag1', 'low_conflict_pct_lag1', 'quad4_pct_lag1']
train:                precision    recall  f1-score   support

  Cooperation       0.96      0.86      0.91       777
High_Conflict       0.69      0.95      0.80        95
 Low_Conflict       0.64      0.81      0.71       172

     accuracy                           0.86      1044
    macro avg       0.77      0.87      0.81      1044
 weighted avg       0.89      0.86      0.87      1044

test:                precision    recall  f1-score   support

  Cooperation       0.92      0.91      0.92       244
High_Conflict       0.55      0.61      0.58        44
 Low_Conflict       0.72      0.70      0.71       108

     accuracy                           0.82       396
    macro avg       0.73      0.74      0.74  

In [ ]:
# 排除 goldstein_min_lag1
features = ['regime_diff', 'event_count_lag1', 'goldstein_std_lag1', 'goldstein_min_lag1', 'num_mentions_sum_lag1',
           'num_articles_sum_lag1', 'num_sources_sum_lag1', 'high_conflict_count_lag1', 'low_conflict_count_lag1',
           'quad4_count_lag1', 'high_conflict_pct_lag1', 'low_conflict_pct_lag1', 'quad4_pct_lag1']
lag_cols_trimmed = [col for col in features if col != 'goldstein_min_lag1']
print(lag_cols_trimmed)

X_trimmed = df[lag_cols_trimmed]
y = df['monthly_label']

X_train_trimmed = X_trimmed[train_mask]
X_test_trimmed = X_trimmed[test_mask]

mod_trimmed = LGBMClassifier(**lgb_params)
mod_trimmed.fit(X_train_trimmed, y_train, sample_weight=compute_sample_weight('balanced', y_train))

pred_train_trimmed = mod_trimmed.predict(X_train_trimmed)
pred_test_trimmed = mod_trimmed.predict(X_test_trimmed)

print('train:', classification_report(y_train, pred_train_trimmed))
print('test:', classification_report(y_test, pred_test_trimmed))


['regime_diff', 'event_count_lag1', 'goldstein_std_lag1', 'num_mentions_sum_lag1', 'num_articles_sum_lag1', 'num_sources_sum_lag1', 'high_conflict_count_lag1', 'low_conflict_count_lag1', 'quad4_count_lag1', 'high_conflict_pct_lag1', 'low_conflict_pct_lag1', 'quad4_pct_lag1']
train:                precision    recall  f1-score   support

  Cooperation       0.96      0.86      0.91       777
High_Conflict       0.69      0.95      0.80        95
 Low_Conflict       0.64      0.81      0.71       172

     accuracy                           0.86      1044
    macro avg       0.77      0.87      0.81      1044
 weighted avg       0.89      0.86      0.87      1044

test:                precision    recall  f1-score   support

  Cooperation       0.92      0.91      0.92       244
High_Conflict       0.55      0.61      0.58        44
 Low_Conflict       0.72      0.70      0.71       108

     accuracy                           0.82       396
    macro avg       0.73      0.74      0.74  

### 模型 B 實驗記錄：regime_diff（政體差異）

- 資料：V-Dem v2x_libdem，取絕對差，年度資料不用lag
- 結果：時間切分驗證下表現變差（HC recall 0.75→0.61），但 LODO 驗證下反而略好（0.461 vs 0.442）
- AUC（模型B）：Macro 0.894 / Cooperation 0.944 / High_Conflict 0.888 / Low_Conflict 0.850
  → 比模型A（Macro 0.917 / HC 0.935）略低，但排序能力仍算穩定
- Gain 顯示 regime_diff 排第一，遠高於其他特徵，懷疑是「認出dyad身份」的捷徑
- 排除 goldstein_min_lag1（gain幾乎0）測試，表現沒變，排除是其他特徵拖累的可能性
- 結論：證據不一致，樣本量（11組dyad）可能是根本限制，先不採用，留待未來擴充dyad後再評估

**目前模型只適用於已知的11組dyad，套用到全新dyad前要重新訓練，不能直接套用**

**因最終方案傾向輸出機率而非強制分類，AUC（排序能力）比recall/precision更該被優先參考**

In [28]:
# Optuna
import optuna
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import recall_score, classification_report
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_sample_weight



def objective(trial):
    params = {
        'objective': 'multiclass',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 4, 31),
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
    }

    tscv = TimeSeriesSplit(n_splits=5)
    scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr, sample_weight=compute_sample_weight('balanced', y_tr))

        y_pred = model.predict(X_val)
        score = recall_score(y_val, y_pred, average='macro', zero_division=0)
        scores.append(score)

    return np.mean(scores)


# ---- 執行搜尋 ----
study = optuna.create_study(direction='maximize', study_name='model_a_tuning')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print('最佳參數:', study.best_params)
print('最佳 CV recall_macro:', study.best_value)

# ---- 用最佳參數，在完整訓練集上重新訓練，並在測試集上驗證 ----
print('\n=== 開始訓練最終模型 ===')
best_params = study.best_params.copy()
best_params.update({
    'objective': 'multiclass',
    'boosting_type': 'gbdt',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
})

final_model = LGBMClassifier(**best_params)
final_model.fit(X_train, y_train, sample_weight=compute_sample_weight('balanced', y_train))

pred_train = final_model.predict(X_train)
pred_test = final_model.predict(X_test)

print('\n=== 調參後最終模型 ===')
print('train:', classification_report(y_train, pred_train))
print('test:', classification_report(y_test, pred_test))



[I 2026-08-17 17:52:01,234] A new study created in memory with name: model_a_tuning
Best trial: 0. Best value: 0.525608:   1%|          | 1/100 [00:02<04:19,  2.62s/it]

[I 2026-08-17 17:52:07,147] Trial 0 finished with value: 0.5256076069650056 and parameters: {'learning_rate': 0.06972428125228906, 'num_leaves': 15, 'max_depth': 5, 'min_child_samples': 14, 'subsample': 0.5888060145272878, 'colsample_bytree': 0.6714375521401117, 'reg_alpha': 0.6890602918807734, 'reg_lambda': 0.46206572623385156, 'n_estimators': 60}. Best is trial 0 with value: 0.5256076069650056.


Best trial: 1. Best value: 0.529412:   2%|▏         | 2/100 [00:03<02:31,  1.55s/it]

[I 2026-08-17 17:52:07,947] Trial 1 finished with value: 0.5294116419345001 and parameters: {'learning_rate': 0.045952845033122365, 'num_leaves': 29, 'max_depth': 2, 'min_child_samples': 31, 'subsample': 0.6809810938432432, 'colsample_bytree': 0.8620046016796271, 'reg_alpha': 0.6410592196873691, 'reg_lambda': 0.7419221105027765, 'n_estimators': 211}. Best is trial 1 with value: 0.5294116419345001.


Best trial: 2. Best value: 0.631338:   3%|▎         | 3/100 [00:04<01:51,  1.15s/it]

...

[I 2026-08-17 17:53:50,822] Trial 98 finished with value: 0.6644702111550511 and parameters: {'learning_rate': 0.01901487700210657, 'num_leaves': 16, 'max_depth': 2, 'min_child_samples': 6, 'subsample': 0.7013003175660778, 'colsample_bytree': 0.8411939035179709, 'reg_alpha': 0.5197599705062297, 'reg_lambda': 0.3366261518757465, 'n_estimators': 56}. Best is trial 98 with value: 0.6644702111550511.


Best trial: 98. Best value: 0.66447:  99%|█████████▉| 99/100 [01:46<00:00,  1.95it/s]

[I 2026-08-17 17:53:51,285] Trial 99 finished with value: 0.6661467795601909 and parameters: {'learning_rate': 0.018909621443467715, 'num_leaves': 13, 'max_depth': 2, 'min_child_samples': 6, 'subsample': 0.6992340850376021, 'colsample_bytree': 0.8203261117853924, 'reg_alpha': 0.4480047925625076, 'reg_lambda': 0.39004263179543947, 'n_estimators': 66}. Best is trial 99 with value: 0.6661467795601909.


Best trial: 99. Best value: 0.666147: 100%|██████████| 100/100 [01:47<00:00,  1.08s/it]


最佳參數: {'learning_rate': 0.018909621443467715, 'num_leaves': 13, 'max_depth': 2, 'min_child_samples': 6, 'subsample': 0.6992340850376021, 'colsample_bytree': 0.8203261117853924, 'reg_alpha': 0.4480047925625076, 'reg_lambda': 0.39004263179543947, 'n_estimators': 66}
最佳 CV recall_macro: 0.6661467795601909

=== 開始訓練最終模型 ===

=== 調參後最終模型 ===
train:                precision    recall  f1-score   support

  Cooperation       0.94      0.81      0.87       777
High_Conflict       0.61      0.89      0.72        95
 Low_Conflict       0.52      0.71      0.60       172

     accuracy                           0.80      1044
    macro avg       0.69      0.81      0.73      1044
 weighted avg       0.84      0.80      0.82      1044

test:                precision    recall  f1-score   support

  Cooperation       0.94      0.85      0.89       244
High_Conflict       0.55      0.59      0.57        44
 Low_Conflict       0.64      0.76      0.69       108

     accuracy                         

In [29]:
import optuna
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_sample_weight
# 目標函數改為 Macro AUC（呼應最終產品採機率輸出，而非強制分類的設計方向）
# 同時將 sample_weight 的加權強度也納入 Optuna 搜尋範圍，而非固定使用 balanced


def objective(trial):
    params = {
        'objective': 'multiclass',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 4, 31),
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
    }

    # 加權強度：1.0 = 完全balanced（原始反比權重），0.0 = 完全不加權，讓 Optuna 自己找中間平衡點
    weight_power = trial.suggest_float('weight_power', 0.0, 1.5)

    tscv = TimeSeriesSplit(n_splits=5)
    scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        base_weights = compute_sample_weight('balanced', y_tr)
        adjusted_weights = base_weights ** weight_power  # 開根號(0.5)~平方(2)之間的軟化/強化調整

        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr, sample_weight=adjusted_weights)

        y_pred_proba = model.predict_proba(X_val)
        classes = model.classes_
        y_val_binarized = label_binarize(y_val, classes=classes)

        # 避免驗證折裡剛好缺某個類別導致 roc_auc_score 出錯
        try:
            score = roc_auc_score(y_val_binarized, y_pred_proba, average='macro', multi_class='ovr')
        except ValueError:
            score = np.nan

        if not np.isnan(score):
            scores.append(score)

    return np.mean(scores) if scores else 0.0


# 執行optuna 
study = optuna.create_study(direction='maximize', study_name='model_b_auc_tuning')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print('最佳參數:', study.best_params)
print('最佳 CV Macro AUC:', study.best_value)

# 用最佳參數，在完整訓練集上重新訓練，並在測試集上驗證 ----
best_params = study.best_params.copy()
weight_power = best_params.pop('weight_power')
best_params.update({
    'objective': 'multiclass',
    'boosting_type': 'gbdt',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
})

final_weights = compute_sample_weight('balanced', y_train) ** weight_power

final_model = LGBMClassifier(**best_params)
final_model.fit(X_train, y_train, sample_weight=final_weights)

pred_train = final_model.predict(X_train)
pred_test = final_model.predict(X_test)

print('\n=== 調參後最終模型（以 AUC 為目標）===')
print('train:', classification_report(y_train, pred_train))
print('test:', classification_report(y_test, pred_test))

# 順便印出最終測試集的 AUC，方便跟之前的版本比較
pred_proba_test = final_model.predict_proba(X_test)
classes = final_model.classes_
y_test_binarized = label_binarize(y_test, classes=classes)

macro_auc = roc_auc_score(y_test_binarized, pred_proba_test, average='macro', multi_class='ovr')
print(f'\nTest Macro AUC: {macro_auc:.3f}')
for i, cls in enumerate(classes):
    auc = roc_auc_score(y_test_binarized[:, i], pred_proba_test[:, i])
    print(f'{cls} AUC: {auc:.3f}')

[I 2026-08-17 17:56:41,431] A new study created in memory with name: model_b_auc_tuning
Best trial: 0. Best value: 0.785224:   1%|          | 1/100 [00:01<03:12,  1.95s/it]

[I 2026-08-17 17:56:43,494] Trial 0 finished with value: 0.7852240883600584 and parameters: {'learning_rate': 0.055370993442038025, 'num_leaves': 20, 'max_depth': 2, 'min_child_samples': 15, 'subsample': 0.7102156970533747, 'colsample_bytree': 0.7889766683766379, 'reg_alpha': 0.8437067752951515, 'reg_lambda': 0.6825779101443652, 'n_estimators': 120, 'weight_power': 0.2818791112474444}. Best is trial 0 with value: 0.7852240883600584.


Best trial: 0. Best value: 0.785224:   2%|▏         | 2/100 [00:03<02:50,  1.74s/it]

[I 2026-08-17 17:56:45,096] Trial 1 finished with value: 0.7629010508044418 and parameters: {'learning_rate': 0.02799536418879848, 'num_leaves': 29, 'max_depth': 6, 'min_child_samples': 29, 'subsample': 0.5363527877079558, 'colsample_bytree': 0.6979261706344124, 'reg_alpha': 0.9649585124819329, 'reg_lambda': 0.3572055065722687, 'n_estimators': 234, 'weight_power': 0.6621256182409025}. Best is trial 0 with value: 0.7852240883600584.


Best trial: 0. Best value: 0.785224:   3%|▎         | 3/100 [00:04<02:17,  1.42s/it]

...

[I 2026-08-17 17:57:48,895] Trial 97 finished with value: 0.8184775725368088 and parameters: {'learning_rate': 0.011596305870478336, 'num_leaves': 4, 'max_depth': 4, 'min_child_samples': 39, 'subsample': 0.6695953975558417, 'colsample_bytree': 0.6270549915293886, 'reg_alpha': 0.48536140790654664, 'reg_lambda': 0.37650531037123225, 'n_estimators': 60, 'weight_power': 0.3277602573442092}. Best is trial 49 with value: 0.8245388595535083.


Best trial: 49. Best value: 0.824539:  99%|█████████▉| 99/100 [01:07<00:00,  2.24it/s]

[I 2026-08-17 17:57:49,317] Trial 98 finished with value: 0.8052531095593614 and parameters: {'learning_rate': 0.013364123461500657, 'num_leaves': 6, 'max_depth': 5, 'min_child_samples': 42, 'subsample': 0.58244612910212, 'colsample_bytree': 0.6767500923357667, 'reg_alpha': 0.2521795063518097, 'reg_lambda': 0.4719656245802416, 'n_estimators': 50, 'weight_power': 0.1002247836821954}. Best is trial 49 with value: 0.8245388595535083.


Best trial: 49. Best value: 0.824539: 100%|██████████| 100/100 [01:08<00:00,  1.46it/s]


[I 2026-08-17 17:57:49,820] Trial 99 finished with value: 0.7981799047486698 and parameters: {'learning_rate': 0.012520692628926548, 'num_leaves': 8, 'max_depth': 6, 'min_child_samples': 48, 'subsample': 0.6271468672905476, 'colsample_bytree': 0.6367296005677622, 'reg_alpha': 0.31936105586744856, 'reg_lambda': 0.41949423463489705, 'n_estimators': 70, 'weight_power': 0.5548797389445341}. Best is trial 49 with value: 0.8245388595535083.
最佳參數: {'learning_rate': 0.010064728575988969, 'num_leaves': 4, 'max_depth': 6, 'min_child_samples': 38, 'subsample': 0.6459455301191607, 'colsample_bytree': 0.6834313822451178, 'reg_alpha': 0.3549677504898402, 'reg_lambda': 0.2740839920994449, 'n_estimators': 64, 'weight_power': 0.3013525939572423}
最佳 CV Macro AUC: 0.8245388595535083

=== 調參後最終模型（以 AUC 為目標）===
train:                precision    recall  f1-score   support

  Cooperation       0.85      0.98      0.91       777
High_Conflict       0.85      0.56      0.68        95
 Low_Conflict       0.83 

In [30]:
import numpy as np
import pandas as pd
from sklearn.metrics import recall_score, precision_score, f1_score

# 使用 Optuna 調完的 final_model，取得測試集的機率預測
y_pred_proba = final_model.predict_proba(X_test)
classes = list(final_model.classes_)

# 針對 High_Conflict 掃描不同門檻
hc_idx = classes.index('High_Conflict')
y_test_hc_binary = (y_test == 'High_Conflict').astype(int)

hc_results = []
for threshold in np.arange(0.10, 0.55, 0.025):
    y_pred_hc = (y_pred_proba[:, hc_idx] >= threshold).astype(int)
    recall = recall_score(y_test_hc_binary, y_pred_hc, zero_division=0)
    precision = precision_score(y_test_hc_binary, y_pred_hc, zero_division=0)
    f1 = f1_score(y_test_hc_binary, y_pred_hc, zero_division=0)
    hc_results.append({'threshold': threshold, 'recall': recall, 'precision': precision, 'f1': f1})

hc_df = pd.DataFrame(hc_results)
print('=== High_Conflict 門檻掃描 ===')
print(hc_df.to_string(index=False))

# 針對 Low_Conflict 掃描不同門檻
lc_idx = classes.index('Low_Conflict')
y_test_lc_binary = (y_test == 'Low_Conflict').astype(int)

lc_results = []
for threshold in np.arange(0.10, 0.55, 0.025):
    y_pred_lc = (y_pred_proba[:, lc_idx] >= threshold).astype(int)
    recall = recall_score(y_test_lc_binary, y_pred_lc, zero_division=0)
    precision = precision_score(y_test_lc_binary, y_pred_lc, zero_division=0)
    f1 = f1_score(y_test_lc_binary, y_pred_lc, zero_division=0)
    lc_results.append({'threshold': threshold, 'recall': recall, 'precision': precision, 'f1': f1})

lc_df = pd.DataFrame(lc_results)
print('\n=== Low_Conflict 門檻掃描 ===')
print(lc_df.to_string(index=False))
best_hc_threshold = hc_df.loc[hc_df['f1'].idxmax()]
best_lc_threshold = lc_df.loc[lc_df['f1'].idxmax()]

print(f'\n建議 High_Conflict 門檻: {best_hc_threshold["threshold"]:.3f} '
      f'(recall={best_hc_threshold["recall"]:.3f}, precision={best_hc_threshold["precision"]:.3f}, f1={best_hc_threshold["f1"]:.3f})')
print(f'建議 Low_Conflict 門檻: {best_lc_threshold["threshold"]:.3f} '
      f'(recall={best_lc_threshold["recall"]:.3f}, precision={best_lc_threshold["precision"]:.3f}, f1={best_lc_threshold["f1"]:.3f})')

=== High_Conflict 門檻掃描 ===
 threshold   recall  precision       f1
     0.100 0.977273   0.273885 0.427861
     0.125 0.886364   0.410526 0.561151
     0.150 0.840909   0.513889 0.637931
     0.175 0.795455   0.538462 0.642202
     0.200 0.750000   0.550000 0.634615
     0.225 0.704545   0.620000 0.659574
     0.250 0.636364   0.622222 0.629213
     0.275 0.568182   0.625000 0.595238
     0.300 0.522727   0.605263 0.560976
     0.325 0.500000   0.666667 0.571429
     0.350 0.500000   0.687500 0.578947
     0.375 0.386364   0.680000 0.492754
     0.400 0.363636   0.666667 0.470588
     0.425 0.204545   0.692308 0.315789
     0.450 0.022727   0.500000 0.043478
     0.475 0.000000   0.000000 0.000000
     0.500 0.000000   0.000000 0.000000
     0.525 0.000000   0.000000 0.000000

=== Low_Conflict 門檻掃描 ===
 threshold   recall  precision       f1
     0.100 1.000000   0.272727 0.428571
     0.125 1.000000   0.272727 0.428571
     0.150 0.925926   0.480769 0.632911
     0.175 0.888889   0.51

### Threshold Tuning 結果

- Optuna改用AUC當目標，weight_power也交給它搜尋 → 找到0.30（接近不加權）
- Test Macro AUC 0.918，比之前手動調參略高
- 但predict()預設0.5門檻下 recall很低（HC 0.36, LC 0.35），因為機率普遍壓低
- 掃描門檻後找到較佳點：
  - HC門檻0.225：recall 0.705, precision 0.620, f1 0.660
  - LC門檻0.275：recall 0.713, precision 0.726, f1 0.720
- 跟之前手動調參版本打平略優，定案採用這組

In [5]:
# retrain
import json
import joblib
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_sample_weight

best_params = {
    'learning_rate': 0.018909621443467715,
    'num_leaves': 13,
    'max_depth': 2,
    'min_child_samples': 6,
    'subsample': 0.6992340850376021,
    'colsample_bytree': 0.8203261117853924,
    'reg_alpha': 0.4480047925625076,
    'reg_lambda': 0.39004263179543947,
    'n_estimators': 66,
    'objective': 'multiclass',
    'boosting_type': 'gbdt',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}
with open('../outputs/para/lgb_01.json', 'w') as f:
    json.dump(best_params, f, indent=2)

weight_power = 0.3013525939572423

final_weights = compute_sample_weight('balanced', y) ** weight_power
 
final_model = LGBMClassifier(**best_params)
final_model.fit(X, y, sample_weight=final_weights)

 
joblib.dump(final_model, '../outputs/models/mod_lgb.pkl')

    

['../outputs/models/mod_lgb.pkl']

In [ ]:
import json

model_config = {
    'features': X_train.columns.tolist(),
    'thresholds': {
        'High_Conflict': 0.225,
        'Low_Conflict': 0.275
    }
}

with open('../outputs/models/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

: 